In [1]:
import os
os.environ["HF_HOME"] = "/rds/general/user/rmw324/home/raels_playground/hugging_face_llms"

import dllm



In [2]:
import torch    
print(torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0) if                    
  torch.cuda.is_available() else "no GPU") 

True 1 Quadro RTX 6000


In [3]:
# Load model & tokenizer
model = dllm.utils.get_model(model_name_or_path="GSAI-ML/LLaDA-8B-Instruct").eval()


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [4]:
tokenizer = dllm.utils.get_tokenizer(model_name_or_path="GSAI-ML/LLaDA-8B-Instruct")
sampler = dllm.core.samplers.MDLMSampler(model=model, tokenizer=tokenizer)
terminal_visualizer = dllm.utils.TerminalVisualizer(tokenizer=tokenizer)

In [5]:
# Ask a question
messages = [[{"role": "user", "content": "What is the square root of 16 + 5 - 9. Please show your working."}]]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True)

outputs = sampler.sample(inputs, return_dict=True)
response = dllm.utils.sample_trim(tokenizer, outputs.sequences.tolist(), inputs)
print(response[0].strip())

#terminal_visualizer.visualize(outputs.histories, rich=True)

To find the square root of 16 + 5 - 9, follow these steps:

1. Calculate the square root of 16:
   \[
   \sqrt{16} = 4
   \]

2. Add 5 to the result:
   \[
   4 + 5 = 9
   \]

3. Subtract 9:
   \[
   9 - 9 = 2
   \]

So, the square root of 16 + 5 - 9 is 2.


In [ ]:
# Same prompt but with remasking enabled
from dllm.core.samplers import MDLMSamplerConfig

remasking_config = MDLMSamplerConfig(
    max_new_tokens=128,
    steps=16,
    block_size=128,
    return_dict=True,
    dynamic_unmasking=True,
    remask_threshold=0.70,
    remask_cooldown=3,
)

messages = [[{"role": "user", "content": "What is the square root of 16 + 5 - 9. Please show your working."}]]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True)

outputs_remasking = sampler.sample(inputs, config=remasking_config)
response_remasking = dllm.utils.sample_trim(tokenizer, outputs_remasking.sequences.tolist(), inputs)
print(response_remasking[0].strip())

#terminal_visualizer.visualize(outputs_remasking.histories, rich=True)

To find
 square root of \( expression6 + 5 -9 \), we first need to simplify the expression inside the square root.




\[ 16 +5 =5


:

\[ 25 - 9 = 16 \]

Now, we find the square root of
 result:

\[ \sqrt{16
 =
4 \


So, the
 root of \( 16 +
5 -
9 \) is 4


In [7]:
class DiffViewer:                                                                                                                                                               
    SHORT = {                                                                                                                                                                   
        "<|startoftext|>":     "‹bos›",                                                                                                                                         
        "<|begin_of_text|>":   "‹bos›",                                                                                                                                         
        "<|endoftext|>":       "‹eos›",                                                                                                                                         
        "<|eot_id|>":          "‹eot›",                                                                                                                                         
        "<|start_header_id|>": "‹[",                                                                                                                                            
        "<|end_header_id|>":   "]›",
    }                                                                                                                                                                           
    TOKEN_WIDTH, TOKENS_PER_LINE = 7, 18
    GREEN, RED, YELLOW, RESET = "\033[42;30m", "\033[41;97m", "\033[43;30m", "\033[0m"                                                                                          
                                                                                                                                                                                
    def __init__(self, histories, tokenizer):                                                                                                                                   
        self.histories  = histories                                                                                                                                             
        self.tokenizer  = tokenizer                                                                                                                                             
        self.mask_id    = tokenizer.mask_token_id
        self.end_hdr_id = tokenizer.convert_tokens_to_ids("<|end_header_id|>")                                                                                                  
        self.step       = -1            # so first next() lands on 0
        self.sep_width  = self.TOKEN_WIDTH * self.TOKENS_PER_LINE + (self.TOKENS_PER_LINE - 1)                                                                                  
                                                                                                                                                                                
    def _render(self, tid):                                                                                                                                                     
        if tid == self.mask_id:                                                                                                                                                 
            s = "_"                                                                                                                                                             
        else:   
            raw = self.tokenizer.decode([tid])
            s = self.SHORT.get(raw, raw).replace("\n", "⏎").replace("\t", "→").strip() or "·"
        return s[:self.TOKEN_WIDTH].ljust(self.TOKEN_WIDTH)                                                                                                                     
                                                                                                                                                                                
    def show(self, step=None):                                                                                                                                                  
        if step is not None:                                                                                                                                                    
            self.step = step
        self.step = max(0, min(self.step, len(self.histories) - 1))
                                                                                                                                                                                
        ids_now  = self.histories[self.step][0].tolist()
        ids_prev = self.histories[self.step - 1][0].tolist() if self.step > 0 else ids_now                                                                                      
                                                                                                                                                                                
        cells = []
        for prev, now in zip(ids_prev, ids_now):                                                                                                                                
            cell = self._render(now)
            if prev != now:
                if   prev == self.mask_id: color = self.GREEN                                                                                                                   
                elif now  == self.mask_id: color = self.RED
                else:                      color = self.YELLOW                                                                                                                  
                cell = f"{color}{cell}{self.RESET}"                                                                                                                             
            cells.append(cell)
                                                                                                                                                                                
        hdr_pos  = [i for i, t in enumerate(ids_now) if t == self.end_hdr_id]                                                                                                   
        boundary = (hdr_pos[1] + 1) if len(hdr_pos) >= 2 else len(ids_now)
                                                                                                                                                                                
        print(f"--- Step {self.step} / {len(self.histories) - 1} ---  "                                                                                                         
            f"green=committed  red=remasked  yellow=swapped")
        for label, start, end in [("prompt", 0, boundary),                                                                                                                      
                                ("response", boundary, len(ids_now))]:                                                                                                        
            chunk = cells[start:end]
            print(f"── {label} " + "─" * max(1, self.sep_width - len(label) - 4))                                                                                               
            for i in range(0, len(chunk), self.TOKENS_PER_LINE):                                                                                                                
                print(" ".join(chunk[i:i + self.TOKENS_PER_LINE]))
                                                                                                                                                                                
    def next(self): self.step += 1; self.show()
    def prev(self): self.step -= 1; self.show()                                                                                                                                 
    def goto(self, n): self.show(step=n)                                                                                                                                        

viewer = DiffViewer(outputs.histories, tokenizer) 

In [8]:
viewer.next() 

--- Step 0 / 128 ---  green=committed  red=remasked  yellow=swapped
── prompt ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
‹bos›   ‹[      user    ]›      ⏎       ⏎       What    is      the     square  root    of      ·       1       6       +       ·       5      
-       ·       9       .       Please  show    your    working .       ‹eot›   ‹[      ass     istant  ]›     
── response ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
⏎       ⏎       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _      
_       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _      
_       _       _       _       _       _       _       _       _       _       _       _       _   

In [9]:
viewer.prev()       # go back one step                                                                                                                                          


--- Step 0 / 128 ---  green=committed  red=remasked  yellow=swapped
── prompt ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
‹bos›   ‹[      user    ]›      ⏎       ⏎       What    is      the     square  root    of      ·       1       6       +       ·       5      
-       ·       9       .       Please  show    your    working .       ‹eot›   ‹[      ass     istant  ]›     
── response ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
⏎       ⏎       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _      
_       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _      
_       _       _       _       _       _       _       _       _       _       _       _       _   

In [10]:
viewer.goto(64)

--- Step 64 / 128 ---  green=committed  red=remasked  yellow=swapped
── prompt ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
‹bos›   ‹[      user    ]›      ⏎       ⏎       What    is      the     square  root    of      ·       1       6       +       ·       5      
-       ·       9       .       Please  show    your    working .       ‹eot›   ‹[      ass     istant  ]›     
── response ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
⏎       ⏎       To      find    the     square  root    of      ·       1       6       +       ·       5       -       ·       9       ,      
follow  these   steps   :       ⏎       ⏎       1       .       _       _       square  _       _       _       _       _       _       _      
_       _       _       _       _       _       _       _       _       _       _       _       _  

In [13]:
# Same prompt with stale-token remasking (Signal A) — opt-in, independent of dynamic_unmasking.
# Trigger: at each step, mask all currently-committed positions and run an extra forward pass.
# If the currently-committed token isn't in the model's top-`stale_topk` for that masked position,
# it's stale and gets remasked. Catches wrong commits the disagreement trigger misses because the
# canvas was generated to be self-consistent with them.
# Cost: ~2x compute per step (one extra forward pass).
from dllm.core.samplers import MDLMSamplerConfig

stale_config = MDLMSamplerConfig(
    max_new_tokens=128,
    steps=128,
    block_size=128,
    return_dict=True,
    stale_remasking=True,
    stale_topk=50,
    remask_cooldown=3,
)

messages = [[{"role": "user", "content": "What is the square root of 16 + 5 - 9. Please show your working."}]]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True)

outputs_stale = sampler.sample(inputs, config=stale_config)
response_stale = dllm.utils.sample_trim(tokenizer, outputs_stale.sequences.tolist(), inputs)
print(response_stale[0].strip())

To find the square root of 16 + 5 - 9, follow these steps:

1. the square root of 16
   \[
   \sqrt{16}
   \[
    - 9 = 2
   \]

So, the square root of 16 + 5 - 9 is 2.


In [12]:
# Step-by-step diff viewer for the stale-remasking run.
# Re-uses the DiffViewer class defined above. Reset and step through:
viewer_stale = DiffViewer(outputs_stale.histories, tokenizer)
viewer_stale.next()

--- Step 0 / 128 ---  green=committed  red=remasked  yellow=swapped
── prompt ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
‹bos›   ‹[      user    ]›      ⏎       ⏎       What    is      the     square  root    of      ·       1       6       +       ·       5      
-       ·       9       .       Please  show    your    working .       ‹eot›   ‹[      ass     istant  ]›     
── response ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
⏎       ⏎       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _      
_       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _       _      
_       _       _       _       _       _       _       _       _       _       _       _       _   